<a href="https://colab.research.google.com/github/ashycoding/Deep-Learning-Labs/blob/main/lab5_DL_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM for Easy Text Prediction

## Purpose of the Lab
- Introduces how LSTM models work with text data
- Focuses on simple, beginner-friendly implementation
- Designed to run quickly in a lab setting

## Learning Goals
- Understand text as sequential data
- Convert text into numerical format (tokenization)
- Create input-output sequences for training
- Train an LSTM model for next-word prediction
- Generate short text using the trained model

## Dataset
- Uses a small custom text corpus
- No external downloads required
- Easy to understand and manage
- Suitable for college lab environments

## Key Takeaway
- Learn the complete pipeline of text prediction using LSTM, from preprocessing to text generation

# Step 1: Import Libraries and Their Roles

import numpy as np  
- Used for numerical operations and array handling

import tensorflow as tf  
- Main deep learning framework used to build and train models

from tensorflow.keras.preprocessing.text import Tokenizer  
- Converts text into numbers by assigning a unique index to each word

from tensorflow.keras.preprocessing.sequence import pad_sequences  
- Makes all sequences the same length by adding padding (zeros)

from tensorflow.keras.models import Sequential  
- Helps build the neural network layer by layer in a simple way

from tensorflow.keras.layers import Embedding, LSTM, Dense  
- Embedding: Converts word indices into dense vectors (word embeddings)  
- LSTM: Learns patterns in sequence data and remembers context  
- Dense: Final layer that predicts the next word

print("TensorFlow version:", tf.__version__)  
- Displays installed TensorFlow version for compatibility check

In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


# Step 2: Create a Small Text Dataset

- Use a small, simple text dataset for quick training  
- Created manually (no external downloads needed)  
- Easy to understand and modify  

- Students can add their own sentences  
- Helps observe how predictions change  

- Focus is on learning, not large data

In [2]:
corpus = [
    "coding improves problem solving skills",
    "python is widely used in data science",
    "machine learning models learn from data",
    "deep learning helps in image recognition",
    "artificial intelligence is transforming industries",
    "neural networks are inspired by the brain",
    "data science combines statistics and programming",
    "practice coding every day to improve",
    "good data leads to better models",
    "algorithms solve complex problems efficiently",
    "lstm networks are good for sequence data",
    "text prediction uses sequential patterns",
    "clean data is important for accuracy",
    "big data requires powerful processing tools",
    "feature engineering improves model performance",
    "supervised learning uses labeled data",
    "unsupervised learning finds hidden patterns",
    "reinforcement learning learns through rewards",
    "python libraries make development faster",
    "training data impacts model quality",
    "testing data evaluates model performance",
    "overfitting reduces model generalization",
    "underfitting fails to capture patterns",
    "optimization improves learning efficiency",
    "gradient descent updates model weights",
    "activation functions introduce nonlinearity",
    "relu is a popular activation function",
    "softmax is used for classification",
    "loss functions measure model error",
    "epochs define training iterations",
    "batch size affects training speed",
    "data preprocessing is an important step",
    "tokenization converts text into numbers",
    "padding ensures equal sequence length",
    "embedding represents words as vectors",
    "sequence models understand context",
    "time series data is sequential",
    "models improve with more training",
    "validation helps tune hyperparameters",
    "cross validation improves reliability",
    "automation speeds up data analysis",
    "visualization helps understand data",
    "graphs show trends clearly",
    "python is beginner friendly",
    "coding requires logical thinking",
    "experiments help in learning concepts",
    "projects build practical skills",
    "consistency leads to mastery",
    "learning never stops in technology",
    "innovation drives the future"
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)

Number of sentences: 50

Sample corpus:
- coding improves problem solving skills
- python is widely used in data science
- machine learning models learn from data
- deep learning helps in image recognition
- artificial intelligence is transforming industries
- neural networks are inspired by the brain
- data science combines statistics and programming
- practice coding every day to improve
- good data leads to better models
- algorithms solve complex problems efficiently
- lstm networks are good for sequence data
- text prediction uses sequential patterns
- clean data is important for accuracy
- big data requires powerful processing tools
- feature engineering improves model performance
- supervised learning uses labeled data
- unsupervised learning finds hidden patterns
- reinforcement learning learns through rewards
- python libraries make development faster
- training data impacts model quality
- testing data evaluates model performance
- overfitting reduces model generalization
- u

# Step 3: Tokenization

- Tokenization converts words into integers  
- Each word gets a unique number based on the dataset vocabulary  

## Example (from our data)
- coding → 1  
- python → 2  
- learning → 3  

- The exact numbers may change depending on how the tokenizer is fitted  

In [3]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)


Vocabulary size: 174

Word index:
{'data': 1, 'is': 2, 'learning': 3, 'model': 4, 'improves': 5, 'in': 6, 'models': 7, 'to': 8, 'training': 9, 'coding': 10, 'python': 11, 'helps': 12, 'for': 13, 'sequence': 14, 'patterns': 15, 'skills': 16, 'used': 17, 'science': 18, 'networks': 19, 'are': 20, 'the': 21, 'improve': 22, 'good': 23, 'leads': 24, 'text': 25, 'uses': 26, 'sequential': 27, 'important': 28, 'requires': 29, 'performance': 30, 'activation': 31, 'functions': 32, 'understand': 33, 'validation': 34, 'problem': 35, 'solving': 36, 'widely': 37, 'machine': 38, 'learn': 39, 'from': 40, 'deep': 41, 'image': 42, 'recognition': 43, 'artificial': 44, 'intelligence': 45, 'transforming': 46, 'industries': 47, 'neural': 48, 'inspired': 49, 'by': 50, 'brain': 51, 'combines': 52, 'statistics': 53, 'and': 54, 'programming': 55, 'practice': 56, 'every': 57, 'day': 58, 'better': 59, 'algorithms': 60, 'solve': 61, 'complex': 62, 'problems': 63, 'efficiently': 64, 'lstm': 65, 'prediction': 66, 'cl

# Step 4: Create Training Sequences

- Each sentence is split into smaller sequences  
- Sequences grow word by word from the start  

## Example (from our data)
Sentence: coding is fun and creative  

Generated sequences:
- coding is  
- coding is fun  
- coding is fun and  
- coding is fun and creative  

## Learning Process
- Input: all words except the last  
- Output: last word  

- The model learns to predict the next word based on previous words

In [4]:
input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    # Convert sentence into list of word indices

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        # Create sequences like:
        # [coding, is], [coding, is, fun], etc.

        input_sequences.append(n_gram_seq)
        # Store each sequence

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")

for seq in input_sequences[:10]:
    print(seq)

Total training sequences: 201

Some sequences before padding:
[10, 5]
[10, 5, 35]
[10, 5, 35, 36]
[10, 5, 35, 36, 16]
[11, 2]
[11, 2, 37]
[11, 2, 37, 17]
[11, 2, 37, 17, 6]
[11, 2, 37, 17, 6, 1]
[11, 2, 37, 17, 6, 1, 18]


# Step 5: Pad the Sequences

- Sequences have different lengths  
- Neural networks require fixed-size input  

- Padding adds zeros at the beginning of shorter sequences  
- Ensures all sequences have the same length  

- This makes the data suitable for training the model

In [5]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])


Maximum sequence length: 7

Padded sequences:
[[ 0  0  0  0  0 10  5]
 [ 0  0  0  0 10  5 35]
 [ 0  0  0 10  5 35 36]
 [ 0  0 10  5 35 36 16]
 [ 0  0  0  0  0 11  2]
 [ 0  0  0  0 11  2 37]
 [ 0  0  0 11  2 37 17]
 [ 0  0 11  2 37 17  6]
 [ 0 11  2 37 17  6  1]
 [11  2 37 17  6  1 18]]



## 6. Split into input (`X`) and output (`y`)

- `X` = all words except the last word
- `y` = the last word to be predicted

We convert `y` into one-hot encoding because this is a multiclass classification problem.


In [6]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (201, 6)
Shape of y: (201, 174)


# Step 7: Build the LSTM Model

## Model Architecture
- Embedding layer: transforms words into meaningful numeric vectors  
- LSTM layer: captures sequence flow and remembers context  
- Dense layer (softmax): outputs probability for the next word  

In [7]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


## 8. Train the model

Since the dataset is very small, we train for more epochs so the model can learn the patterns better.


In [8]:
# Train the model and store history
history = model.fit(X, y, epochs=200, verbose=1)

# Print final training accuracy
print("Final Training Accuracy:", history.history['accuracy'][-1])

Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.0249 - loss: 5.1588
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0597 - loss: 5.1506
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0597 - loss: 5.1406 
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0597 - loss: 5.1200
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597 - loss: 5.0682
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0597 - loss: 4.9112
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597 - loss: 4.7948
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0597 - loss: 4.7394
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597 - loss: 4.7245
Epoch 10/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597 - loss: 4.7005    
Epoch 11/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597 - loss: 4.7043
Epoch 12/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0597


## 9. Function for next-word text generation

This function:
1. takes a seed text
2. converts it into tokens
3. pads it
4. predicts the next word
5. appends the predicted word to the sentence


In [13]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text



## 10. Test the model

Try a few seed texts and observe the output.


In [10]:
print(generate_text("coding improves", 2))
print(generate_text("python is widely", 2))
print(generate_text("machine learning models", 2))

coding improves problem solving
python is widely used in
machine learning models learn from


In [11]:
# Try different starting texts
print(generate_text("deep", 2))
print(generate_text("models", 2))
print(generate_text("python", 2))

deep learning helps
models understand data
python is beginner
